# **PyTorch Tensor Basics**

### **Laboratory Exercise 4** - *Deep Learning Fundamentals*
*From the course module:* ***02_PyTorch Basics*** *(Laboratory Task 5)*

**Instruction:** Create and manipulate tensors in PyTorch: set a seed, convert a NumPy array to a tensor, change its dtype and shape, index and square it, and compute a matrix product.

1. Perform Standard Imports
2. Create a function called `set_seed()` that accepts `seed: int` as a parameter, this function must return nothing but just set the seed to a certain value.
3. Create a NumPy array called "arr" that contains 6 random integers between 0 (inclusive) and 5 (exclusive), call the `set_seed()` function and use `42` as the seed parameter.
4. Create a tensor "x" from the array above
5. Change the dtype of x from `int32` to `int64`
6. Reshape `x` into a 3x2 tensor.** There are several ways to do this. 
7. Return the right-hand column of tensor `x`
8. Without changing x, return a tensor of square values of `x`.** There are several ways to do this.
9. Create a tensor `y` with the same number of elements as `x`, that can be matrix-multiplied with `x`.** Use PyTorch directly (not NumPy) to create a tensor of random integers between 0 (inclusive) and 5 (exclusive). Use 42 as seed. Think about what shape it should have to permit matrix multiplication.
10. Find the matrix product of `x` and `y`.

## **Overview**

A **tensor** is PyTorch's core data structure: an n-dimensional array (like a NumPy array) that can also run on a GPU and track gradients for backpropagation. The operations in this task are the building blocks of every network in the earlier tasks.

| Rank | Name | Example shape |
|---|---|---|
| 0 | Scalar | `()` |
| 1 | Vector | `(6,)` |
| 2 | Matrix | `(3, 2)` |

Two rules matter for this task:

- **Matrix multiplication:** a $(m \times n)$ tensor can be multiplied with an $(n \times p)$ tensor, and the result is $(m \times p)$.
- **Matching dtypes:** `torch.matmul` requires both tensors to have the same dtype, so mixing `int32` and `int64` raises an error.

## **Implementation**



### **Perform Standard Imports**

In [1]:
!pip install numpy torch

In [2]:
import random

import numpy as np
import torch

### **Create the `set_seed()` Function**

A random number generator produces the same sequence every time it starts from the same seed, which makes results reproducible. NumPy, PyTorch, and Python's `random` module each keep their own generator, so the function seeds all three. It returns nothing.

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

### **Create the NumPy Array `arr`**

`set_seed(42)` is called **before** generating the numbers. `np.random.randint(0, 5, 6)` draws 6 integers from 0 (inclusive) to 5 (exclusive).

The dtype is set to `int32` explicitly. NumPy's default integer type differs by platform and version (often `int64`), and the next steps assume the array starts as `int32`.

In [4]:
set_seed(42)

arr = np.random.randint(0, 5, size=6, dtype=np.int32)

print("arr  :", arr)
print("dtype:", arr.dtype)

arr  : [3 4 2 4 4 1]
dtype: int32


### **Create the Tensor `x` from the Array**

`torch.from_numpy()` converts the array to a tensor. The tensor **shares memory** with the array, so changing one changes the other. Use `torch.tensor(arr)` instead if an independent copy is needed.

In [5]:
x = torch.from_numpy(arr)

print("x    :", x)
print("dtype:", x.dtype)

x    : tensor([3, 4, 2, 4, 4, 1], dtype=torch.int32)
dtype: torch.int32


### **Change the dtype from `int32` to `int64`**

`.type(torch.int64)` returns a new tensor with the requested dtype. Equivalent options are `x.to(torch.int64)` and `x.long()`.

In [6]:
x = x.type(torch.int64)

print("x    :", x)
print("dtype:", x.dtype)

x    : tensor([3, 4, 2, 4, 4, 1])
dtype: torch.int64


### **Reshape `x` into a 3×2 Tensor**

`.reshape(3, 2)` arranges the 6 elements into 3 rows and 2 columns, filling row by row. Other ways that give the same result:

In [7]:
x = x.reshape(3, 2)

# Equivalent alternatives (each gives the same 3x2 tensor):
assert torch.equal(x, x.view(3, 2))
assert torch.equal(x, x.view(-1, 2))          # -1 lets PyTorch infer the size
assert torch.equal(x, torch.reshape(x, (3, 2)))

print(x)
print("shape:", x.shape)

tensor([[3, 4],
        [2, 4],
        [4, 1]])
shape: torch.Size([3, 2])


### **Return the Right-Hand Column of `x`**

Slicing with `[:, -1]` means "all rows, last column". `-1` counts from the end, so it selects the right-hand column no matter how many columns there are.

In [8]:
right_col = x[:, -1]

print(right_col)
print("shape:", right_col.shape)

# Keep it as a 3x1 column instead of a flat vector:
print(x[:, -1:])

tensor([4, 4, 1])
shape: torch.Size([3])
tensor([[4],
        [4],
        [1]])


### **Return the Squares of `x` Without Changing `x`**

Each of these creates a **new** tensor and leaves `x` untouched. The in-place version `x.pow_(2)` (trailing underscore) would modify `x`, so it is avoided here.

In [9]:
x_squared = x ** 2

# Equivalent alternatives:
assert torch.equal(x_squared, torch.square(x))
assert torch.equal(x_squared, x.pow(2))
assert torch.equal(x_squared, x * x)

print("x squared:\n", x_squared)
print("x unchanged:\n", x)

x squared:
 tensor([[ 9, 16],
        [ 4, 16],
        [16,  1]])
x unchanged:
 tensor([[3, 4],
        [2, 4],
        [4, 1]])


### **Create Tensor `y` for Matrix Multiplication**

`x` has 6 elements and shape $(3 \times 2)$. For $x \cdot y$ to be defined, the inner dimensions must match, so `y` needs 2 rows. To also have 6 elements it must have 3 columns:

$$
(3 \times \mathbf{2})\cdot(\mathbf{2} \times 3) = (3 \times 3)
$$

`y` is created directly with `torch.randint(0, 5, (2, 3))` after resetting the seed to 42. `torch.randint` returns `int64` by default, which matches `x` (this is why step 5 converted `x` to `int64`).

In [10]:
set_seed(42)

y = torch.randint(0, 5, (2, 3))

print(y)
print("shape:", y.shape)
print("dtype:", y.dtype)

tensor([[2, 2, 1],
        [4, 1, 0]])
shape: torch.Size([2, 3])
dtype: torch.int64


### **Find the Matrix Product of `x` and `y`**

The `@` operator performs matrix multiplication. Equivalent calls are `torch.matmul(x, y)` and `torch.mm(x, y)`. Each entry is a row of `x` dotted with a column of `y`, for example:

$$
(x\,y)_{11} = (3)(2) + (4)(4) = 22
$$

In [11]:
product = x @ y

# Equivalent alternatives:
assert torch.equal(product, torch.matmul(x, y))
assert torch.equal(product, torch.mm(x, y))

print(product)
print("shape:", product.shape)

tensor([[22, 10,  3],
        [20,  8,  2],
        [12,  9,  4]])
shape: torch.Size([3, 3])


## **Final Answer**

| Step | Result |
|---|---|
| 3. `arr` (seed 42) | `[3 4 2 4 4 1]`, dtype `int32` |
| 4. `x` | `tensor([3, 4, 2, 4, 4, 1], dtype=torch.int32)` |
| 5. `x` as `int64` | `tensor([3, 4, 2, 4, 4, 1])` |
| 6. `x` reshaped to 3×2 | `[[3, 4], [2, 4], [4, 1]]` |
| 7. Right-hand column | `tensor([4, 4, 1])` |
| 8. `x` squared | `[[9, 16], [4, 16], [16, 1]]` |
| 9. `y` (2×3, seed 42) | `[[2, 2, 1], [4, 1, 0]]` |
| 10. `x @ y` | `[[22, 10, 3], [20, 8, 2], [12, 9, 4]]` |

**Interpretation**

Setting the seed before each random step made every result reproducible, so the numbers above are the same on every run. The 6 values from NumPy were converted to a tensor, widened to `int64`, and reshaped into a 3×2 matrix. Indexing and squaring returned new tensors without changing `x`. To multiply, `y` had to be 2×3, so that the inner dimensions match and the result is 3×3.

**Conclusion:**

Tensors behave like NumPy arrays but add dtype control and GPU/autograd support. Getting the dtype and shape right before an operation, as in steps 5 and 9, is what prevents the most common PyTorch errors.